# Star Formation

This notebook shows how to render **grid-based snapshots** from Athena++:

QuickStart:

```text
AMR snapshot -> export_volume_vdb_sequence() -> .vdb file -> setup_animation() in Blender
```

Use this when the source data is already volume data, for example Athena++ or another AMR code.


## 0. Configuration Check

Download the following Dataset if you do not have the dataset for example:

```bash
git lfs fetch --exclude="*"
git lfs pull --include="Example/Star_Formation/Data/star_formation_test_012*.athdf"
```

Before running, please ensure your environment, dependencies, and local data streams are correctly configured. You may run the following test:

In [ ]:
#1. Library Test
try:
    import numpy as np
    import numba
    import yt
    import openvdb
    
    print(f"[SUCCESS] NumPy version: {np.__version__} (Expected: 2.2.5)")
    print(f"[SUCCESS] Numba version: {numba.__version__}")
    print(f"[SUCCESS] yt version: {yt.__version__}")
    
    version = getattr(openvdb, "__vdb_version__", getattr(openvdb, "LIBRARY_VERSION", "Detected"))
    print(f"[SUCCESS] OpenVDB version: {version}")
    
except ImportError as e:
    print(f"[FAILED] Missing library: {e}")
    import sys
    sys.exit(1)

#2. Dataset Test
import os
target_dir = "./Data/"
if os.path.exists(target_dir):
    files = [f for f in os.listdir(target_dir) if f.startswith("star_formation_test_012") and f.endswith(".athdf")]
    if files:
        print(f"Found {len(files)} targeted dataset files in {target_dir}")
        # Verify LFS pointer is fully replaced by the real binary file
        if all(os.path.getsize(os.path.join(target_dir, files[i])) > 1024 * 1024 for i in range(len(files))):  # Check if any file is larger than 1MB
            print(f"[SUCCESS] File storage check passed (Smallest File Size = {min(os.path.getsize(os.path.join(target_dir, f)) for f in files)/(1024*1024):.2f} MB). Git LFS pull succeeded.")
        else:
            print("[FAILED] File exists but is an un-fetched Git LFS pointer text (< 1MB). Please re-run git lfs pull.")
    else:
        print(f"  [FAILED] No matching star_formation_test_012*.athdf files found in {target_dir}")
else:
    print(f"  [FAILED] Target directory {target_dir} does not exist.")


## 1. Export VDB frames

Run this section in a Python 3.11 environment with `yt` and `openvdb` available.


In [ ]:
import sys
sys.path.append("../../..") ## Set it to the path of the AstroVis package if you are running this script outside of the package directory. 

from AstroVis.backend import export_volume_vdb_sequence

start_frame = 120
end_frame = 129
input_dir = "Data"
output_dir = "Output"

export_volume_vdb_sequence(input_dir=input_dir, output_dir=output_dir, object_name="Star_Formation",
                           start_frame=start_frame, end_frame=end_frame,
                            vtype="gas", field="density", log=True,
                            preview_dir="Plots",preview_every=1, scale=100,
                            zip_output=False, delete_unzipped=False,)



In [ ]:
# Check ds.derived_field_list first when you want to confirm the yt particle type and field name.

import os
import yt

snapshots = sorted([f for f in os.listdir(input_dir) if f.endswith(".athdf")])
ds = yt.load(os.path.join(input_dir, snapshots[0]))
print(ds.derived_field_list)

## 2. Import and render in Blender

Run this section **inside Blender's Python environment**.  
Opening 'Scripting' in the Blender, add a new text block and paste the following script in the text block.  
Press 'Run' button.

In [ ]:
import bpy

import sys
sys.path.append(r"../../..") 
from AstroVis.api import *

import numpy as np

vdb_path = "Output"

scene = SceneManager()
scene.clear_scene(use_empty=False)
scene.set_camera(location=(0.3, 0.3, 0.3), look_at=(0, 0, 0))

object_name = "Star_Formation"

# Create a volume material for the field data
material = create_field_volume_material(
    object_name,
    field_min=1,  # Set the minimum field value for visualization
    field_max=5,
    emission_multiplier=100,
    apply=False,
)[object_name]

setup_animation(data_path=vdb_path,
                object_material = {object_name: material})


scene.set_world_background(color=(0.0, 0.0, 0.0, 1.0), strength=0.0)

scene = bpy.context.scene
scene.render.engine = 'CYCLES'

scene.cycles.device = 'GPU'                    # confirm OptiX/CUDA is selected in prefs
scene.cycles.samples = 32                       # plenty for a structure check
scene.cycles.use_denoising = False              # denoising can hide real detail at low res
scene.cycles.use_adaptive_sampling = True
scene.cycles.adaptive_threshold = 0.05

scene.cycles.max_bounces = 32
scene.cycles.transparent_max_bounces = 24
scene.cycles.volume_bounces = 2
scene.cycles.volume_step_rate = 1.0             # coarser steps = big speedup
scene.cycles.volume_max_steps = 256              # 1024 was overkill for a test render

scene.render.resolution_percentage = 50          # half-res for a fast preview

for obj in bpy.context.scene.objects:
    if obj.type == 'VOLUME':
        volume_data = obj.data
        
        obj.data.display.interpolation_method = 'CUBIC'

        volume_data.render.space = 'WORLD'
        volume_data.render.step_size = 0.1


You may run the bash script blender.sh directly for rendering figures in headless mode. 
Just change the path of blender to where you install:

```bash
GALLIUM_DRIVER=llvmpipe ../../../code/blender-4.2.0-linux-x64/blender -b --factory-startup -E CYCLES --python Star_Formation_Blender.py -o "//Blender_Plot/frame_" -s 0 -e 4 -a
```

And check the figures in Blender_Plot.

## Notes

- `ds.field_list` is the first thing to check when you are unsure about the yt field type or the field name.
- `export_volume_sequence()` is the simple one-pass helper: it exports every frame, saves preview images every 10th frame by default, and returns the final global range for Blender.
- Keep the same `field_min` and `field_max` across all frames so the Blender material uses one consistent scale.
- The examples here intentionally use only the minimum useful arguments. For optional controls, see `Documentation/Backend/Volume_Data.md`, `Documentation/Blender/Animation_setup.md`, and `Documentation/Blender/Node_and_shading.md`.
- `setup_volume_animation()` is the lower-level alternative when you want to import the VDB sequence directly instead of using `setup_animation()`.
- The volume material already emits light from the data values, so this example keeps the scene simple and does **not** add extra lights.


## Detail API

1. loads each snapshot with `yt.load()`
2. calls `load_volume()` to build the AMR hierarchy
3. computes the field range and preview slice via `analyze_field_data()`
4. calls `hierarchy_to_multiple_vdbs()` to export each frame to VDB files (`frame_0000/`, `frame_0001/`, ...)
5. saves previews every `preview_every` frames using the final global range
6. prints the final global min/max so the Blender material can reuse the same scale